<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Real tests at an online shop

**[Real tests at an online shop](https://learning.nextia-ai.com/courses/experiments/m09/real-tests-at-an-online-shop/)** · Experiments and Causal Impact: Did Our Change Work? · Case study 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** check and decide real online tests from their daily summary numbers: the split check, the result at the planned look, what daily peeking would have declared, a sequential test that allows early stops, and how many winners luck alone gives over many tests. You compare the rules fairly and run the finished check on tests that you set aside.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook runs in under a minute |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | the interval and the p-value, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/); the split check, from [Check the test before the result](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/); [Peeking](https://learning.nextia-ai.com/courses/experiments/m04/peeking/); and [Many metrics and many tests](https://learning.nextia-ai.com/courses/experiments/m04/many-metrics-and-many-tests/). |
| **Data** | Real: the ASOS Digital Experiments Dataset, CC BY 4.0 (credit below). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m09/real-tests-at-an-online-shop/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M09-L02-real-tests-at-an-online-shop/real-tests-at-an-online-shop-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the data

Run the next cell. It downloads `asos_digital_experiments_dataset.csv` (2.7 MB) from the Open Science Framework (OSF) into a folder `asos-experiments`, and checks its checksum. It needs an internet connection and no account. If a copy of the file is next to this notebook, the cell uses that copy.

You should see `Ready: asos_digital_experiments_dataset.csv, checksum OK.`

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `asos-experiments` (in Colab: the **Files** panel on the left), then run the cell again. If it happens again, the file on OSF has changed: compare it with the dataset card in this folder.

The data is the **ASOS Digital Experiments Dataset** by C. H. Bryan Liu, Ângelo Cardoso, Paul Couturier and Emma J. McCoy (ASOS.com and Imperial College London, 2021), [osf.io/64jsb](https://osf.io/64jsb/), DOI 10.17605/OSF.IO/64JSB, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). This notebook downloads the original file unchanged; the analysis drops 8 rows that have no users yet. The data is real: it comes from 78 tests that ran on a real online shop. It has no rows for single people: every number is a total over many thousands of users.

In [ ]:
# Setup: download the ASOS file from OSF into asos-experiments/ and check it. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

URL = "https://osf.io/download/hq8sc/"
FILE, SHA256 = "asos_digital_experiments_dataset.csv", "aa0cc3f8304fac545f232ec0bc363373f932919f6e72e28908958bc10651ce22"
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a folder with a copy of the file
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("asos-experiments").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()
path = WORK / FILE

def sha256(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

if not (path.exists() and sha256(path) == SHA256):
    if (HERE / FILE).exists():
        shutil.copy(HERE / FILE, path)
    elif LOCAL and (Path(LOCAL) / FILE).exists():
        shutil.copy(Path(LOCAL) / FILE, path)
    else:
        try:
            urllib.request.urlretrieve(URL, path)
        except OSError:
            # Python without certificates (python.org on macOS): try curl.
            if subprocess.run(["curl", "-fsSL", "-o", str(path), URL]).returncode:
                raise SystemExit(f"Could not download {URL}. Check your internet connection.")
if sha256(path) != SHA256:
    raise SystemExit(f"{FILE} has the wrong checksum. Delete the folder asos-experiments and run again.")
os.chdir(WORK)
print(f"Ready: {FILE}, checksum OK.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. (The Larkfield notebooks also had `truth()`; real data has no true effects, so it is not here.). They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the case study's functions

The next cell defines the functions that the lesson page explains: the result of one row (`add_result`), the plan's decision rule (`decision`), the split check (`srm_p`, `split_check`), one look a day (`daily_looks`), the course's sequential test (`sequential_path`) and Benjamini–Hochberg. Run it. It prints nothing.

In [ ]:
# The case study's functions. The sequential test and Benjamini-Hochberg are copies of the course project's code.
import math
KEY = ["experiment_id", "variant_id"]


def add_result(rows):
    """Difference (treatment - control), standard error, 95% interval and p-value of every row, from the summary
    statistics alone (a normal approximation: every group has many thousands of users)."""
    r = rows.copy()
    r["difference"] = r["mean_t"] - r["mean_c"]
    r["se"] = np.sqrt(r["variance_t"] / r["count_t"] + r["variance_c"] / r["count_c"])
    r["low"], r["high"] = r["difference"] - 1.96 * r["se"], r["difference"] + 1.96 * r["se"]
    r["relative"] = r["difference"] / r["mean_c"]
    r["p"] = 2 * stats.norm.sf(np.abs(r["difference"]) / r["se"])
    return r


def decision(low, high):
    """The plan's rule at the planned look."""
    return "ship" if low > 0 else "do not ship" if high < 0 else "no clear effect"


def srm_p(count_c, count_t, share_t=0.5):
    """Chi-square test of the users per group against the planned split (as xp srm)."""
    total = count_c + count_t
    return float(stats.chisquare([count_c, count_t], [total * (1 - share_t), total * share_t]).pvalue)


def split_check(count_c, count_t):
    """'not checkable' (more than 0.5 points from 50%: ask for the planned split), 'mismatch' or 'ok'."""
    if abs(count_t / (count_c + count_t) - 0.5) > 0.005:
        return "not checkable"
    return "mismatch" if srm_p(count_c, count_t) < 0.001 else "ok"


def daily_looks(series):
    """One look a day: for each whole day, the latest row at or before it; the last row is always a look."""
    t = series["time_since_start"].to_numpy()
    days = range(int(math.ceil(t.min() - 1e-9)), int(t.max()) + 1)
    picks = {int(np.nonzero(t <= day + 1e-9)[0].max()) for day in days if (t <= day + 1e-9).any()}
    return series.iloc[sorted(picks | {len(t) - 1})].reset_index(drop=True)


def always_valid_halfwidth(v, tau2, alpha=0.05):
    return math.sqrt(v * (v + tau2) / tau2 * (2 * math.log(1 / alpha) + math.log((v + tau2) / v)))


def always_valid_p(diff, v, tau2):
    """1 / (the mixture likelihood ratio against 'no effect'), capped at 1."""
    lr = math.sqrt(v / (v + tau2)) * math.exp(diff**2 * tau2 / (2 * v * (v + tau2)))
    return min(1.0, 1.0 / lr)


def sequential_path(diffs, variances, tau2, alpha=0.05):
    """The always-valid intervals and p-values along the looks (running intersection and minimum)."""
    lo, hi, pmin, out = -np.inf, np.inf, 1.0, []
    for d, v in zip(diffs, variances):
        h = always_valid_halfwidth(v, tau2, alpha)
        lo, hi = max(lo, d - h), min(hi, d + h)
        pmin = min(pmin, always_valid_p(d, v, tau2))
        out.append((lo, hi, pmin))
    return out


def benjamini_hochberg(p, alpha=0.05):
    """True for the discoveries at a false discovery rate of alpha."""
    p = np.asarray(p)
    m = len(p)
    order = np.argsort(p)
    below = p[order] <= alpha * np.arange(1, m + 1) / m
    keep = np.zeros(m, bool)
    if below.any():
        keep[order[: np.nonzero(below)[0].max() + 1]] = True
    return keep

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data, briefly

Each row is one **comparison** (one treatment against its control), one metric and one moment of the test. The numbers are **cumulative**: each row covers every user from the start of the test to that moment.

Run the cell. It loads the file, drops the rows that have no users yet, and counts the experiments and comparisons.

In [ ]:
tests = pd.read_csv("asos_digital_experiments_dataset.csv", dtype={"experiment_id": str})
print(tests.shape)
print("rows with no users yet:", int(((tests.count_c == 0) | (tests.count_t == 0)).sum()))
tests = tests[(tests.count_c > 0) & (tests.count_t > 0)]
tests = tests.sort_values(KEY + ["metric_id", "time_since_start"]).reset_index(drop=True)
print("experiments:", tests.experiment_id.nunique(), "| comparisons:", len(tests[KEY].drop_duplicates()))
print("rows per metric:", tests.metric_id.value_counts().sort_index().to_dict())
print("rows without a variance:", tests.groupby("metric_id").variance_c.apply(lambda x: int(x.isna().sum())).to_dict())

> **Check.** You should see `(24153, 10)`, 8 rows with no users yet, 78 experiments and 99 comparisons. Metric 1 has no missing variance, so it is complete.

Look at one comparison of metric 1 (a share of users: 0 or 1 per user). The counts grow, and so does the mean: each user's value counts everything that the user did since the start.

In [ ]:
one = tests[(tests.experiment_id == "036afc") & (tests.variant_id == 2) & (tests.metric_id == 1)]
print(one[["time_since_start", "count_c", "count_t", "mean_c", "mean_t"]].iloc[[0, 1, 2, -1]].to_string(index=False))

> **Check.** You should see four rows: days 1.5, 2.0, 2.5 and 76.5. The control group grows from 188,065 users to 1,050,994.

## 2. Set aside next month's tests

Before any result, put 8 experiments aside. You choose every rule on the other 70, and you run the finished check on these 8 once, at the end.

In [ ]:
ids = sorted(tests.experiment_id.unique())
holdout = sorted(np.random.default_rng(2021).choice(ids, 8, replace=False).tolist())
past = tests[~tests.experiment_id.isin(holdout)]
print("set aside:", holdout)
print("past tests:", past.experiment_id.nunique(), "experiments,", len(past[KEY].drop_duplicates()), "comparisons")

> **Check.** You should see 8 experiment IDs, starting with `530a76`, and 70 experiments with 90 comparisons.

## 3. Check the split first

Take the last look of every comparison on metric 1, and compute the share of users in the treatment group.

In [ ]:
m1 = add_result(past[past.metric_id == 1])
last = m1.groupby(KEY).tail(1).reset_index(drop=True)
first = m1.groupby(KEY).head(1).reset_index(drop=True)
last["share_first"] = first.count_t / (first.count_c + first.count_t)
last["share_t"] = last.count_t / (last.count_c + last.count_t)
last["srm_p"] = [srm_p(c, t) for c, t in zip(last.count_c, last.count_t)]
last["split"] = [split_check(c, t) for c, t in zip(last.count_c, last.count_t)]
print(last.split.value_counts().to_dict())
print(last[last.split != "ok"][KEY + ["count_c", "count_t", "share_first", "share_t", "srm_p", "split"]]
      .sort_values("share_t").to_string(index=False))

> **Check.** You should see `{'ok': 80, 'not checkable': 8, 'mismatch': 2}` and a table of 10 comparisons.

Eight shares are far from 50%, and each one is the same at the first look and the last. That looks like a planned unequal split (2 : 1, or 33 : 34 in a test with three groups), not a bug. The plan is not published, so we cannot check them: they get no decision. Two shares are close to 50% but too far for chance with so many users: a sample ratio mismatch.

> **Your turn.** Compute the split check's p-value for comparison `26bd38`, variant 1, against 50%, with `srm_p`. Put it in `srm_26bd38`.

In [ ]:
srm_26bd38 = ...  # srm_p(count_c, count_t) of experiment 26bd38, variant 1, at its last look

In [ ]:
expect('srm_26bd38', srm_26bd38, 6.5439e-08, tolerance=0.001)

## 4. The result at the planned look

For the 80 comparisons with a good split, apply the plan's rule to the last look: interval above 0, ship; below 0, do not ship; otherwise, no clear effect.

In [ ]:
ok = last[last.split == "ok"].copy()
ok["decision"] = [decision(lo, hi) for lo, hi in zip(ok.low, ok.high)]
print(ok.decision.value_counts().to_dict())
print(f"median size of a change: {ok.relative.abs().median():.2%} of the control mean")
print(f"median standard error: {(ok.se / ok.mean_c).median():.2%} of the control mean")
print(f"median length: {ok.time_since_start.median():.0f} days; median users: {(ok.count_c + ok.count_t).median():,.0f}")

> **Check.** You should see `{'no clear effect': 59, 'ship': 12, 'do not ship': 9}`. The changes are tiny: a median of 0.18% of the control mean.

> **Predict.** If none of the 80 changes did anything, how many would show p < 0.05 by luck? How many do you expect here?

The next cell counts the p-values in ten bins. With no effect at all, p-values spread evenly: about 8 per bin.

In [ ]:
import matplotlib.pyplot as plt
counts, _ = np.histogram(ok.p, bins=np.linspace(0, 1, 11))
print("p-values per bin of 0.1:", counts.tolist())
plt.bar(np.arange(0.05, 1, 0.1), counts, width=0.09, color="#0054D8")
plt.axhline(len(ok) / 10, color="#B54708", linestyle="--", label="if no change did anything")
plt.xlabel("p-value at the planned look"); plt.ylabel("comparisons"); plt.legend(); plt.show()

> **Check.** You should see `[26, 9, 4, 6, 5, 4, 13, 4, 1, 8]`. The first bin is far above the line: some changes did something.

Tests with a real effect give small p-values. So p-values above 0.5 come almost only from tests with **no** effect, and those spread evenly from 0 to 1. Count the share above 0.5 and double it: that estimates the share of tests with no effect.

> **Your turn.** Put the estimated share of comparisons with no effect in `null_share`: the share of `ok.p` above 0.5, times 2, at most 1. Round it to 3 decimals.

In [ ]:
null_share = ...  # the share of p-values above 0.5, doubled, at most 1, rounded to 3 decimals

In [ ]:
expect('null_share', null_share, 0.75)

## 5. Many tests: how many winners are luck?

The next cell compares the count of p < 0.05 with what luck alone gives, and applies the two corrections from [Many metrics and many tests](https://learning.nextia-ai.com/courses/experiments/m04/many-metrics-and-many-tests/).

In [ ]:
n = len(ok)
print("p < 0.05:", int((ok.p < 0.05).sum()), "| luck alone, if no change worked:", 0.05 * n)
no_effect = min(1.0, float((ok.p > 0.5).mean()) * 2)  # the share of no-effect tests, as above
print("expected false winners, with the share of no-effect tests:", round(0.05 * no_effect * n, 1))
print("Bonferroni:", int((ok.p < 0.05 / n).sum()), "| Benjamini-Hochberg:", int(benjamini_hochberg(ok.p).sum()))

> **Check.** You should see `p < 0.05: 21`, then 3.0 expected false winners, Bonferroni 11 and Benjamini–Hochberg 16.

## 6. What daily peeking would have declared

Now the time dimension. For every comparison, take one look a day and find the first day each rule would stop. `stops()` uses the course's sequential test (mSPRT) with `tau`, its guess of the size of a real effect, as a share of the control mean.

In [ ]:
def first_day(looks, mask):
    """The day of the first look where mask is true, or None."""
    mask = np.asarray(mask, bool)
    return float(looks["time_since_start"].to_numpy()[mask.argmax()]) if mask.any() else None


def stops(looks, tau_rel):
    """When would each rule stop this comparison? looks: one comparison's daily looks with add_result columns."""
    tau2 = (tau_rel * looks["mean_c"].iloc[-1]) ** 2
    seq_p = np.array([s[2] for s in sequential_path(looks["difference"], looks["se"] ** 2, tau2)])
    now_p = np.array([always_valid_p(d, v, tau2) for d, v in zip(looks["difference"], looks["se"] ** 2)])
    confirmed = (now_p < 0.05) & np.concatenate([[False], now_p[:-1] < 0.05])  # two looks in a row
    sign = lambda mask: None if not np.any(mask) else (
        "up" if looks["difference"].to_numpy()[np.argmax(mask)] > 0 else "down")
    return {"looks": len(looks), "end_day": float(looks["time_since_start"].iloc[-1]),
            "peek_day": first_day(looks, looks["p"] < 0.05), "peek_sign": sign(looks["p"] < 0.05),
            "seq_day": first_day(looks, seq_p < 0.05), "seq_sign": sign(seq_p < 0.05),
            "conf_day": first_day(looks, confirmed), "conf_sign": sign(confirmed)}


def all_stops(m1, comparisons, tau_rel):
    """stops() for every comparison in the list, joined to its decision at the planned look."""
    rows = []
    for e, v in comparisons[KEY].itertuples(index=False):
        looks = daily_looks(m1[(m1.experiment_id == e) & (m1.variant_id == v)])
        rows.append({"experiment_id": e, "variant_id": v, **stops(looks, tau_rel)})
    return pd.DataFrame(rows).merge(comparisons[KEY + ["decision"]], on=KEY)

> **Predict.** Peeking stops at the first day with p < 0.05. Of the 80 comparisons, how many will it stop with a "winner" or a "loser"? The planned look decided 21.

In [ ]:
paths = all_stops(m1, ok, tau_rel=0.05)
peek = paths[paths.peek_day.notna()]
print("peeking calls:", len(peek), "of", len(paths), "| median day of the call:", peek.peek_day.median())
print("of these, no clear effect at the planned look:", int((peek.decision == "no clear effect").sum()))

> **Check.** You should see `peeking calls: 35 of 80`, a median day of 7, and 14 calls that end with no clear effect.

The next chart shows one of them, `b382c6`: peeking called a winner on day 2, and the test ended with p = 0.69.

In [ ]:
looks = daily_looks(m1[(m1.experiment_id == "b382c6") & (m1.variant_id == 1)])
tau2 = (0.05 * looks.mean_c.iloc[-1]) ** 2
seq_p = [s[2] for s in sequential_path(looks.difference, looks.se ** 2, tau2)]
plt.plot(looks.time_since_start, looks.p, label="usual p-value at that day")
plt.plot(looks.time_since_start, seq_p, label="sequential (always-valid) p-value")
plt.axhline(0.05, color="#B54708", linestyle="--", label="0.05")
plt.xlabel("day of the test"); plt.ylabel("p-value"); plt.legend(); plt.show()
print(looks[["time_since_start", "relative", "p"]].iloc[:4].round(4).to_string(index=False))

> **Check.** You should see the usual p-value dip below 0.05 on day 2, then rise. The sequential p-value never comes near 0.05.

## 7. A sequential test, and the size of tau

The course's default tau is 5% of the control mean. These changes are much smaller. A better guess comes from past tests: the spread of the estimated changes, minus the spread that noise alone gives.

In [ ]:
rel, rel_se = ok.relative.to_numpy(), (ok.se / ok.mean_c).to_numpy()
tau_past = float(np.sqrt(np.var(rel) - np.mean(rel_se ** 2)))
print(f"spread of the estimates {np.std(rel):.2%}, of the noise {np.sqrt(np.mean(rel_se ** 2)):.2%}: tau {tau_past:.2%}")
for name, tau in [("tau 5%", 0.05), ("tau from past tests", tau_past)]:
    s = all_stops(m1, ok, tau)
    seq = s[s.seq_day.notna()]
    print(f"{name}: {len(seq)} calls, median day {seq.seq_day.median()}, "
          f"{int((seq.end_day - seq.seq_day).sum())} test-days saved")

> **Check.** You should see tau about 0.59%, then 15 calls with tau 5% and 18 with tau from past tests.

## 8. A failure case: a glitch fools the sequential test

Two of the sequential test's calls end with no clear effect. One of them is `162a38`, on day 38. The dataset's own datasheet warns about this test: on day 38 the treatment mean jumps, probably because some tracking data was lost.

In [ ]:
paths_past = all_stops(m1, ok, tau_past)
print(paths_past[(paths_past.seq_day.notna()) & (paths_past.decision == "no clear effect")].to_string(index=False))
g = m1[(m1.experiment_id == "162a38") & (m1.variant_id == 1)]
print(g[(g.time_since_start >= 37) & (g.time_since_start <= 39.5)][["time_since_start", "mean_c", "mean_t", "relative", "p"]]
      .round(5).to_string(index=False))

> **Check.** You should see two comparisons: `162a38` (stopped on day 38) and `54a85a` variant 2 (stopped on day 7). In the second table, the change jumps on day 38 and falls back.

A sequential test keeps its promise only if the data is right. One bad look is enough, because once a value is out of the interval, it stays out. The fix is cheap: **confirm** the stop. Stop only if the look's own always-valid p-value is below 0.05 at two looks in a row. A real effect stays; a glitch goes away. `stops()` already computes it: `conf_day`.

In [ ]:
conf = paths_past[paths_past.conf_day.notna()]
agree = ((conf.conf_sign == "up") & (conf.decision == "ship")) | ((conf.conf_sign == "down") & (conf.decision == "do not ship"))
print("confirmed early stops:", len(conf), "| same answer as the planned look:", int(agree.sum()),
      "| median day:", conf.conf_day.median())
print(conf[~agree].to_string(index=False))

> **Check.** You should see 17 confirmed early stops, 16 with the same answer as the planned look, and one row: `54a85a` variant 2.

The glitch is gone. The one disagreement, `54a85a`, is the other test that the datasheet names: its treatment group rises faster than its control, which may be a **primacy effect**, the opposite of novelty (see [Novelty and learning effects](https://learning.nextia-ai.com/courses/experiments/m04/novelty-and-learning-effects/)). Early in the test, users needed time to get used to the change.

## 9. The fair comparison: rebuilt A/A tests

The real tests have no known truth, so we cannot count false winners in them. We can rebuild each comparison as an **A/A test**: the same looks and the same standard errors, but no effect at all. Then we count how often each rule calls a winner or a loser.

In [ ]:
def aa_versions(se, mean_c, tau_rel, n, rng):
    """n A/A versions of one comparison: its own looks and standard errors, and no effect at all.
    Returns, for each rule, which versions it would have called a winner or a loser."""
    info = np.maximum.accumulate(1 / np.asarray(se, float) ** 2)  # information = 1 / se squared
    steps = np.diff(np.concatenate([[0.0], info]))
    diff = (rng.standard_normal((n, len(info))) * np.sqrt(steps)).cumsum(axis=1) / info  # a random walk
    v = 1 / info
    p = 2 * stats.norm.sf(np.abs(diff) / np.sqrt(v))

    def seq_p(tau):  # the always-valid p-value at each look, with tau as a share of the control mean
        t2 = (tau * mean_c) ** 2
        return np.minimum(1, np.sqrt((v + t2) / v) * np.exp(-diff**2 * t2 / (2 * v * (v + t2))))

    planned, below = p[:, -1] < 0.05, seq_p(tau_rel) < 0.05
    confirmed = (below[:, 1:] & below[:, :-1]).any(axis=1)
    return {"one planned look": planned, "peeking every day": (p < 0.05).any(axis=1),
            "sequential, tau 5%": (seq_p(0.05) < 0.05).any(axis=1), "sequential, tau from past tests": below.any(axis=1),
            "confirmed sequential": confirmed, "confirmed early stop, else planned look": confirmed | planned}

> **Predict.** Which rules will stay under 5% false winners?

In [ ]:
rng = np.random.default_rng(40)
rates = {}
for e, v in ok[KEY].itertuples(index=False):
    looks = daily_looks(m1[(m1.experiment_id == e) & (m1.variant_id == v)])
    calls = aa_versions(looks.se.to_numpy(), looks.mean_c.iloc[-1], tau_past, 1000, rng)
    for rule, called in calls.items():
        rates.setdefault(rule, []).append(called.mean())
aa = pd.Series({rule: np.mean(r) for rule, r in rates.items()}).round(3)
print(aa.to_string())

> **Check.** You should see about 4.9% for one planned look, about 17% for peeking every day, and less than 1% for the sequential tests. The last rule stays under 5%.

The numbers move a little with the seed: 1,000 A/A versions of 80 comparisons give 80,000 A/A tests.

> **Your turn.** The rebuilt A/A tests say that peeking every day calls a winner in about 17% of tests with no effect. Of the 35 comparisons that peeking called, how many would have been called by the plan's own rule at the planned look? Put the count in `peek_confirmed`.

In [ ]:
peek_confirmed = ...  # peeking calls (paths.peek_day not missing) whose planned-look decision is not "no clear effect"

In [ ]:
expect('peek_confirmed', peek_confirmed, 21)

## 10. The finished check on next month's tests

The recommended rule: check the split first; stop early only with the **confirmed** sequential test (tau from past tests); otherwise decide at the planned look. `review()` does all of it for new tests. Run it once on the 8 experiments that you set aside.

In [ ]:
def review(rows, tau_rel):
    """The finished check for new tests, one row per comparison: split check, then a confirmed early stop if
    there is one, else the decision at the planned look."""
    m = add_result(rows[rows.metric_id == 1])
    out = []
    for (e, v), series in m.groupby(KEY):
        looks = daily_looks(series)
        end = looks.iloc[-1]
        split = split_check(end.count_c, end.count_t)
        s = stops(looks, tau_rel)
        if split != "ok":
            verdict, day = f"no decision ({split})", None
        elif s["conf_day"] is not None:
            verdict, day = ("ship" if s["conf_sign"] == "up" else "do not ship") + " (early)", s["conf_day"]
        else:
            verdict, day = decision(end.low, end.high), s["end_day"]
        out.append({"experiment_id": e, "variant_id": v, "users": int(end.count_c + end.count_t),
                    "treatment share": round(end.count_t / (end.count_c + end.count_t), 4),
                    "change at the end": f"{end.relative:+.2%}", "p at the end": round(end.p, 4),
                    "verdict": verdict, "decided on day": day, "planned length (days)": s["end_day"]})
    return pd.DataFrame(out)

In [ ]:
new = tests[tests.experiment_id.isin(holdout)]
result = review(new, tau_past)
print(result.to_string(index=False))

> **Check.** You should see 9 comparisons: 2 with no decision (a mismatch), 2 ships, 1 do not ship and 4 with no clear effect; two of the decisions came early.

> **Your turn.** How many test-days did the early stops save on these 8 experiments? Add, over the comparisons decided early, the planned length minus the day of the decision. Put the total in `days_saved`.

In [ ]:
days_saved = ...  # sum of (planned length - decided on day) over the verdicts that end with "(early)"

In [ ]:
expect('days_saved', days_saved, 51.5)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m09/real-tests-at-an-online-shop/) explains each step with pictures, gives the recommendation and its limits, and shows how to make this case study a portfolio piece. Its knowledge checks count toward your certificate.

**Credit.** The data is the **ASOS Digital Experiments Dataset** by C. H. Bryan Liu, Ângelo Cardoso, Paul Couturier and Emma J. McCoy (ASOS.com and Imperial College London, 2021), [osf.io/64jsb](https://osf.io/64jsb/), DOI 10.17605/OSF.IO/64JSB, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). This notebook downloads the original file unchanged; the analysis drops 8 rows that have no users yet.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [A minimum wage without a test](https://learning.nextia-ai.com/courses/experiments/m09/a-minimum-wage-without-a-test/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m09/real-tests-at-an-online-shop/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and aggregated: it has no rows for single people.